<a href="https://colab.research.google.com/github/edwardoughton/satellite-image-analysis/blob/main/05_01_ggs416_26_02_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🛰️ GGS416 Week 5: Image Classification (NAIP) 🛰️

## Why this week matters
This lesson is the bridge between segmentation (where we were grouping pixels by similarity) and classification (where we will assign semantic labels).

By the end of class, you should be able to go from raw NAIP imagery to a fully evaluated supervised classification map.


## Learning objectives

This week you should be able to:
1. Explain segmentation vs classification in remote sensing workflows.
2. Distinguish supervised and unsupervised classification.
3. Build a multi-feature raster stack (R, G, B, NDVI, NDWI, Brightness).
4. Train a Random forest classifier on labeled pixels.
5. Evaluate performance with train/test split and confusion matrix.
6. Interpret where and why model errors occur.

# An introduction to classification

Classification is a machine learning task that assigns an input sample to one of several predefined categories (e.g., target classes).

Formally, the approach is based on training a model to learn a mapping like so:

$f$: $X$ → $Y$

Where:
- X = input feature space  
- Y = finite set of discrete class labels  

In other words, the model learns a function $𝑓$ that maps input features $𝑋$ to class labels $Y$.

The goal is to learn decision boundaries that separate classes in the feature space.

Here, we are aiming to predict discrete categories (vegetation, water, concrete etc.).

We are not aiming to predict continuous values, as we would use other statistical tools for that purpose (e.g., regression methods).

### Examples

* Binary classification - Two mutually exclusive classes (built-up pixel versus non-built-up pixel).
* Multiclass classification - More than two mutually exclusive classes (urban, suburban, forest, desert, etc.).
* Multilabel classification - An instance can belong to multiple classes at the same time (images tagged as "rural", "fields" and "wheat").


## Supervised versus unsupervised approaches

You may hear this dichotomy regularly.

Supervised classification:
- Requires labeled training data.
- Learns from (input, label) pairs.
- Examples: Logistic Regression, Support Vector Machine (SVM), Neural Networks, etc..

Unsupervised classification:
- No labeled data.
- Groups samples using clustering.
- Example: k-means.



## Processing pipeline workflow

The general approach to the workflow today (and in general) is as follows:

* Data collection  
* Feature extraction or representation learning  
* Model training  
* Prediction on new data  
* Model evaluation  




## Common classification methods

- Logistic regression  
- k-Nearest Neighbors (kNN)  
- Support Vector Machines (SVM)  
- Decision trees  
- Random forests  
- Neural networks  

Today we will explore decision trees and random forests.

## Evaluation metrics

Evaluation metrics quantify how well a classification model performs by comparing predicted labels to true labels.

- Accuracy - The proportion of total predictions that are correct, calculated as correctly classified samples divided by all samples.
- Precision - The proportion of predicted positive instances that are actually positive, measuring the model’s positive predictive value.
- Recall  - The proportion of actual positive instances that are correctly identified, measuring the model’s completeness.
- F1-score  - The harmonic mean of precision and recall, providing a single measure that balances both.
- Confusion matrix  - A tabular summary of prediction outcomes showing true positives, true negatives, false positives, and false negatives.


## Concepts checkpoint

Before we progress further, you should remember that:

- Segmentation is the partitioning of image spaces into coherent regions.
- Classification is the assignment of class labels (water, vegetation, built, etc.).
- Unsupervised methods (e.g., k-means) help to discover structure without labels.
- Supervised methods help to learn a mapping from labeled examples.

Today we will cover supervised classification at the pixel level, to get started.

### Exercise: Concept sort
Classify each statement as Segmentation, Classification, or Both:
1. "Groups neighboring pixels with similar appearance."
2. "Assigns semantic labels like water or vegetation."
3. "Can be unsupervised."
4. "Can be supervised."

Then explain one sentence for your hardest choice.

### Enter your answers here



## Why random forests

Before we move into classification, we need to understand the model we are aiming to use, e.g., a random forest classifier.

A random forest is a machine learning algorithm that:

- Builds many individual decision trees.
- Each tree then makes a classification decision.
- The final class is determined by majority vote.

You can think of it like asking many simple decision-makers:

> “Is this pixel vegetation?”  
> “Is this pixel water?”  
> “Is this pixel built?”

Each tree makes a prediction, and the forest votes on the final answer, across all individual trees.



### Decision trees

A decision tree works by asking a sequence of yes/no questions about feature values.

For example:

- Is NDVI > 0.4?
    - If yes, then likely vegetation  
    - If no, then check brightness  
- Is brightness > 0.6?
    - If yes, then likely built surface  

A tree keeps splitting the data based on feature thresholds until it assigns a class label.



### Why use random forest classification for satellite imagery?

Random forest is widely used in remote sensing because:

1. It handles multiple features easily (R, G, B, NDVI, NDWI, Brightness, etc.)
2. It captures nonlinear relationships
3. It is relatively robust to noise
4. It performs well even with limited training data
5. It requires minimal parameter tuning

Importantly, the approach does not require classes to be perfectly separable by a single threshold, making it well suited for real-world satellite imagery.


### What exactly is the model learning?

The model learns patterns such as:

- High NDVI + moderate brightness is most likely vegetation  
- High NDWI + low NIR is most likely water  
- High brightness + low NDVI is most likely built surfaces  

It learns these relationships automatically from labeled training data.

So rather than manually defining thresholds, the random forest learns rules from data during training, and can swiftly combine multiple features simultaneously.

Instead of choosing one threshold at a time, we allow the model to discover patterns in feature space.


### Transition from segmentation to classification

Segmentation grouped similar pixels.

Classification assigns meaning to those pixels.

We are now shifting from manually defined rules, to data-driven learning from labeled examples.

Let us now get into an example.

## Installing necessary packages
This cell installs only what we need for this class, based on the packages we have already covered earlier in the semester:
- `pystac-client`, `planetary-computer`, `requests` - NAIP download from STAC
- `rasterio` - geospatial raster IO and clipping
- `scikit-learn` - Random forest model and metrics
- `matplotlib`, `seaborn` - visualization

If your environment already has these packages, skip this step.

In [ ]:
#!pip -q install pystac-client planetary-computer requests rasterio scikit-learn matplotlib seaborn

## Package imports and notebook setup
We import all libraries at once so students can see the full set of dependencies before we get stuck into the coding.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import requests
from pystac_client import Client
import planetary_computer as pc

import rasterio
from rasterio.windows import from_bounds

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, classification_report)

## Reuse prior NAIP download function (continuity with earlier classes)
To reduce cognitive load, we keep the same function style used previously.

This reinforces that new analytical ideas can build on stable code you are already familiar with.

### Inputs
- `bbox`: area of interest in lon/lat
- `datetime`: date range string
- `directory`: local output folder

### Output
- Local path to one NAIP GeoTIFF

In [ ]:
# Same style/function name used in Week 3 notebook

def my_download_function(bbox, datetime, directory):
    os.makedirs(directory, exist_ok=True)

    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    item = next(
        catalog.search(
            collections=["naip"],
            bbox=bbox,
            datetime=datetime,
            limit=1,
            method="POST",
        ).items()
    )

    href = pc.sign(item).assets["image"].href
    filename = os.path.basename(href.split("?")[0])
    out_path = os.path.join(directory, filename)

    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print("Already downloaded:", out_path)
    else:
        with requests.get(href, stream=True) as r:
            r.raise_for_status()
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print("Saved:", out_path)

    return out_path

## Download one NAIP image (e.g., for Northern Virginia)
We use NAIP because it has a high spatial resolution and is visually intuitive when classifying pixels.

In [ ]:
# Keep date range aligned with NAIP availability on Planetary Computer
bbox = (-77.70, 38.50, -76.80, 39.10)
datetime = "2023-06-01/2023-09-28"
directory = "naip_downloads"

naip_path = my_download_function(bbox, datetime, directory)
print("Using file:", naip_path)

## Reuse prior raster clipping pattern
This step narrows the tile to a smaller teaching AOI so:
- training is faster,
- plots are clearer,
- classroom runtime is predictable.

We keep the same `rasterio` window workflow from prior lectures.

In [ ]:
# Same rasterio workflow as previous, but auto-centers a smaller teaching AOI

os.makedirs("naip_cropped", exist_ok=True)
out_path = os.path.join("naip_cropped", f"crop_{os.path.basename(naip_path)}")

with rasterio.open(naip_path) as src:
    print("CRS:", src.crs)

    b = src.bounds
    # Make a smaller center crop to keep runtime low in class
    x_margin = (b.right - b.left) * 0.43
    y_margin = (b.top - b.bottom) * 0.43

    win = from_bounds(
        b.left + x_margin,
        b.bottom + y_margin,
        b.right - x_margin,
        b.top - y_margin,
        src.transform,
    )
    # print(win)
    data = src.read(window=win)

    prof = src.profile
    prof.update(
        height=data.shape[1],
        width=data.shape[2],
        transform=rasterio.windows.transform(win, src.transform),
    )

with rasterio.open(out_path, "w", **prof) as dst:
    dst.write(data)

print("Wrote:", out_path)

## Visualize RGB
This visual baseline helps students sanity-check the scene before modeling.

### Exercise: Pre-classification visual audit
- Identify three land-cover types you can clearly see in the RGB image.
- For each type, write one reason it may be easy or hard to classify at pixel level.
- Predict one pair of classes that might be confused later.

In [ ]:
path_in = os.path.join("naip_cropped", f"crop_{os.path.basename(naip_path)}")

with rasterio.open(path_in) as src:
    img = src.read().astype("float32")

rgb = img[:3]
rgb01 = rgb / (rgb.max() + 1e-6)
rgb01 = np.transpose(rgb01, (1, 2, 0))

plt.figure(figsize=(8, 8))
plt.imshow(rgb01)
plt.title("NAIP RGB (Cropped)")
plt.axis("off")
plt.show()

## Build a feature stack: R, G, B, NDVI, NDWI, Brightness

To train a classifier like a random forest, we cannot give it an image as a picture.

Instead, we must represent the image numerically.

Each pixel becomes a feature vector, so a list of measurable values describing that pixel.

For pixel $i$, we define:

$$
\mathbf{x}_i = [R_i, G_i, B_i, NDVI_i, NDWI_i, Brightness_i]
$$

This means:

- $R_i$ = red reflectance value at pixel $i$

- $G_i$ = green reflectance value at pixel $i$

- $B_i$ = blue reflectance value at pixel $i$

- $NDVI_i$ = vegetation index value at pixel $i$

- $NDWI_i$ = water index value at pixel $i$

- $Brightness_i$ = average reflectance across RGB bands at pixel $i$

This is a key transition, so going from the image as a picture to image as a machine learning data structure.

### Spectral Indices

Spectral indices combine different wavelength bands to highlight specific surface properties.

#### Normalized Difference Vegetation Index (NDVI)

$$
NDVI = \frac{NIR - R}{NIR + R + \epsilon}
$$

where:

- $NIR$ = Near Infrared band  
- $R$ = Red band  
- $\epsilon$ = small constant to prevent division by zero  

High NDVI values typically indicate healthy vegetation because vegetation reflects strongly in the near-infrared and absorbs red light.

#### Normalized Difference Water Index (NDWI)

$$
NDWI = \frac{G - NIR}{G + NIR + \epsilon}
$$

where:

- $G$ = Green band  
- $NIR$ = Near Infrared band  
- $\epsilon$ = small constant to prevent division by zero  

High NDWI values typically indicate water bodies because water absorbs strongly in the near-infrared while reflecting green light.

#### Brightness

$$
Brightness = \frac{R + G + B}{3}
$$

where:

- $R$ = Red band  
- $G$ = Green band  
- $B$ = Blue band  

Brightness represents overall reflectance intensity and is often useful for identifying bright surfaces such as impervious areas.


### Machine learning perspective

After building the feature stack, the image is no longer treated as a picture.

Instead, it becomes a table where:

- Each row represents one pixel
- Each column represents one feature

If the image contains 1,000,000 pixels, then the dataset looks like this:

| Pixel ID | R | G | B | NDVI | NDWI | Brightness |
|-----------|---|---|---|------|------|------------|
| 1         | 0.21 | 0.34 | 0.18 | 0.62 | -0.10 | 0.24 |
| 2         | 0.55 | 0.57 | 0.52 | 0.12 | -0.35 | 0.55 |
| 3         | 0.03 | 0.05 | 0.08 | -0.10 | 0.72 | 0.05 |
| ...       | ... | ... | ... | ... | ... | ... |
| N         | ... | ... | ... | ... | ... | ... |

The classifier does not "see" an image.  

It sees a numerical matrix:

- Rows = pixels  
- Columns = features  

This is a key transition, hence with the image going from a visible picture to a usable dataset for machine learning purposes.

In [ ]:
# Example: Make the feature stack
r = img[0]
g = img[1]
b = img[2]
nir = img[3]

ndvi = (nir - r) / (nir + r + 1e-6)
ndwi = (g - nir) / (g + nir + 1e-6)
brightness = (r + g + b) / 3.0

feature_stack = np.dstack([r, g, b, ndvi, ndwi, brightness])
feature_names = ["R", "G", "B", "NDVI", "NDWI", "Brightness"]

print("Feature stack shape (rows, cols, features):", feature_stack.shape)
print(feature_stack[:1])

### Exercise - Feature reasoning before modeling
Before running training, make a hypothesis table:
- Which feature should best separate water from non-water?
- Which feature should best separate vegetation from built?
- Which feature might be noisy or redundant?

Keep your predictions so you can compare with feature importance later.

In [ ]:
# Example: Visualize pixel values
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
im0 = ax[0].imshow(ndvi, cmap="RdYlGn")
ax[0].set_title("NDVI")
ax[0].axis("off")
fig.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

im1 = ax[1].imshow(ndwi, cmap="Blues")
ax[1].set_title("NDWI")
ax[1].axis("off")
fig.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)

im2 = ax[2].imshow(brightness, cmap="gray")
ax[2].set_title("Brightness")
ax[2].axis("off")
fig.colorbar(im2, ax=ax[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Create training labels
For teaching speed, let us generate provisional labels from thresholds.

This is not the final scientific labeling standard, but it is excellent for demonstrating the full supervised pipeline in one class period.

Class IDs:
- `0`: water
- `1`: vegetation
- `2`: built/bright
- `255`: unlabeled/ignored

In [ ]:
# Example: Set class IDs:
# 0 = water
# 1 = vegetation
# 2 = built/bright
# 255 = unlabeled (ignore)

labels = np.full(ndvi.shape, 255, dtype=np.uint8)

water_mask = ndwi > 0.10
veg_mask = (ndvi > 0.25) & (~water_mask)
built_mask = (brightness > np.percentile(brightness, 70)) & (ndvi < 0.20) & (~water_mask)

labels[water_mask] = 0
labels[veg_mask] = 1
labels[built_mask] = 2

class_names = {0: "water", 1: "vegetation", 2: "built"}

print("Labeled pixels:", int((labels != 255).sum()))
for cid, cname in class_names.items():
    print(f"{cname:>10}:", int((labels == cid).sum()))

In [ ]:
# Visual check of pseudo-labels
label_vis = np.zeros((*labels.shape, 3), dtype=np.float32)
label_colors = {
    0: np.array([0.1, 0.5, 1.0]),
    1: np.array([0.1, 0.8, 0.2]),
    2: np.array([1.0, 0.2, 0.2]),
}
for cid, color in label_colors.items():
    label_vis[labels == cid] = color

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(rgb01)
ax[0].set_title("RGB")
ax[0].axis("off")

ax[1].imshow(rgb01)
ax[1].imshow(label_vis, alpha=0.45)
ax[1].set_title("Pseudo-label Overlay")
ax[1].axis("off")
plt.show()

### Exercise - Label quality check
Inspect the pseudo-label overlay and answer:
- Where do labels look clearly correct?
- Where do you see obvious label noise?
- Which class is most sensitive to threshold choices?

Extension: adjust one threshold (e.g., NDWI 0.10 to 0.05 or 0.15) and observe what changes most.

## Random forest method

A random forest is an ensemble learning method composed of many decision trees.

Each tree:
- Is trained on a bootstrap sample (random sample with replacement) of the training data.
- Uses only a random subset of features at each split.
- Produces its own class prediction.

The final prediction is obtained by majority vote across all trees.

## A single decision tree

In classification trees, splits are selected to reduce node impurity (e.g., how mixed the classes are in a node).

A common impurity measure is the Gini index:

$$
Gini(t) = 1 - \sum_{k=1}^{K} p(k|t)^2
$$

Where:

- $t$ = current node  
- $ K $ = total number of classes  
- $ p(k|t) $ = proportion (probability) of samples in node $ t $ that belong to class $ k $  

Interpretation:
- $ Gini(t) = 0 $ A Gini index of 0 means the node is pure, so all samples belong to one class.
- Higher values mean more class mixing in the node.

### Split quality

When a node $ t $ is split into two child nodes $ t_L $ (left) and $ t_R $ (right), we measure the decrease in impurity:

$$
\Delta Gini = Gini(t) - \left(\frac{N_L}{N} Gini(t_L) + \frac{N_R}{N} Gini(t_R)\right)
$$

Where:

- $ N $ = number of samples in parent node $ t $  
- $ N_L $ = number of samples in left child node  
- $ N_R $ = number of samples in right child node  
- $ Gini(t_L), Gini(t_R) $ = impurity of the child nodes  

A split is preferred when $ \Delta Gini $ is large (i.e., impurity decreases significantly).

## Forest prediction

Majority voting is achieved as follows.

Suppose we have:

- $ B $ = total number of trees in the forest  
- $ \mathbf{x} $ = feature vector for one sample  
- $ h_b(\mathbf{x}) $ = prediction of tree $ b $ for input $ \mathbf{x} $  

The final predicted class $ \hat{y} $ is:

$$
\hat{y} = \mathrm{mode}\{h_b(\mathbf{x})\}_{b=1}^{B}
$$

Where:

- $ \mathrm{mode} $ = the most frequently predicted class among all trees  




### Advantages of random forests

Importantly, random forests:

- Handle nonlinear decision boundaries.
- Are robust to mixed feature scales (e.g., spectral bands, indices).
- Can be tolerant to correlated features.
- Reduce overfitting compared to a single tree.
- Provide interpretable outputs such as feature importance and confusion matrix evaluation.

## Sample training data and split train/test
We flatten raster pixels into rows, keep only labeled pixels, sample a balanced set, then split:

- train set - fit model
- test set - estimate generalization

This avoids evaluating on the same data used for fitting.

Here we will deliberately inject extremely large noise to illustrate how sensitive supervised models are to training data.

In [ ]:
# Example: Split training data. X is feature matrix. Y is target vector.
rng = np.random.default_rng(42)

X_all = feature_stack.reshape(-1, feature_stack.shape[-1])
y_all = labels.reshape(-1)

valid_idx = np.where(y_all != 255)[0]
X_valid = X_all[valid_idx]
y_valid = y_all[valid_idx]

# Balanced sampling for classroom stability
samples_per_class = 250
chosen = []
for cid in sorted(class_names):
    idx = np.where(y_valid == cid)[0]
    take = min(samples_per_class, len(idx))
    chosen.append(rng.choice(idx, size=take, replace=False))

chosen = np.concatenate(chosen)
X = X_valid[chosen]
y = y_valid[chosen]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# For the sake of this example, add artificial training noise
X_train = X_train + np.random.normal(0.5, 0.25, X_train.shape)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## Train the random forest classifier
Key parameters to discuss in class:
- `n_estimators`: number of trees (more trees usually stabilize predictions)
- `max_depth`: tree depth control (regularization)
- `random_state`: reproducibility for instruction

In [ ]:
# Example: Train our random forest

n_estimators = 5
max_depth = 5
random_state = 42

rf = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=random_state,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

print("Model trained.")

## Evaluate performance
We use:
- Accuracy: - fraction of correct predictions
- Classification report -  precision/recall/F1 per class
- Confusion matrix -  where classes are confused

Confusion matrix supports error diagnosis much better than a single scalar score.

In [ ]:
# Example: Produce our performance metrics
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.3f}")
print()
print(classification_report(y_test, y_pred, target_names=[class_names[i] for i in sorted(class_names)]))

cm = confusion_matrix(y_test, y_pred, labels=sorted(class_names))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=[class_names[i] for i in sorted(class_names)]).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

### Exercise - Error diagnosis from confusion matrix

After running the evaluation you should:

- Identify the most frequent misclassification pair.
- For that pair, give two plausible spectral reasons for confusion.
- Suggest one new feature (or data source) that might reduce this error.

## Predict classes for a full scene and visualize
After validation, we apply the classifier to every pixel feature vector and map predictions back to image space.

In [ ]:
# Example: Get our full prediction for the whole scene
import matplotlib.patches as mpatches

full_pred = rf.predict(X_all).reshape(labels.shape)

pred_vis = np.zeros((*full_pred.shape, 3), dtype=np.float32)
for cid, color in label_colors.items():
    pred_vis[full_pred == cid] = color

fig, ax = plt.subplots(1, 3, figsize=(16, 6))
ax[0].imshow(rgb01)
ax[0].set_title("RGB")
ax[0].axis("off")

ax[1].imshow(pred_vis)
ax[1].set_title("Predicted Classes")
ax[1].axis("off")

ax[2].imshow(rgb01)
ax[2].imshow(pred_vis, alpha=0.45)
ax[2].set_title("Prediction Overlay")
ax[2].axis("off")

# Add legend
legend_handles = []
for cid, class_name in class_names.items():
    color = label_colors[cid]
    patch = mpatches.Patch(color=color, label=class_name)
    legend_handles.append(patch)

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=len(class_names),
    frameon=True
)

plt.tight_layout()
plt.show()

# Exercise - Exploring the impact of noise

When we developed the training data, for this stylized class example, we explicitly added noise to our data (e.g., `X_train = X_train + np.random.normal(25, 50, X_train.shape)`).

* Run the model again with no noise and interpret the results.
* Explore the sensitivity of noise on the full scene prediction.  

## Interpret feature importance

Random forest feature importance (impurity-based) gives a practical first pass on which variables drive splits most often.

Class discussion prompt: does this ranking match domain expectations (e.g., NDWI for water, NDVI for vegetation)?

In [ ]:
# Example: Understand random forest feature importance
importances = rf.feature_importances_
order = np.argsort(importances)[::-1]

plt.figure(figsize=(7, 4))
plt.bar(np.array(feature_names)[order], importances[order])
plt.title("Random forest Feature Importance")
plt.ylabel("Importance")
plt.show()

### Exercise - Importance versus hypothesis
Compare your earlier exercise predictions with observed importance:
- Which expectations matched?
- Which did not?
- If a feature seems "too important," is that physical signal, class balance, or label leakage?

## Discussion
1. Where did we find misclassifications concentrated spatially?
2. Which class pairs are most confused in the confusion matrix?
3. How much do NDVI and NDWI improve separability?
4. Which errors are likely from weak labels vs weak features?
5. How would an object based workflow change this process?

## References
1. Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5-32. https://doi.org/10.1023/A:1010933404324
2. Ho, T. K. (1995). Random Decision Forests. *Proceedings of 3rd ICDAR*, 278-282. https://doi.org/10.1109/ICDAR.1995.598994
3. Belgiu, M., & Dragut, L. (2016). Random forest in remote sensing: A review of applications and future directions. *ISPRS Journal of Photogrammetry and Remote Sensing*, 114, 24-31. https://doi.org/10.1016/j.isprsjprs.2016.01.011
4. Rodriguez-Galiano, V. F., et al. (2012). An assessment of the effectiveness of a random forest classifier for land-cover classification. *ISPRS Journal of Photogrammetry and Remote Sensing*, 67, 93-104. https://doi.org/10.1016/j.isprsjprs.2011.11.002
5. Maxwell, A. E., Warner, T. A., & Fang, F. (2018). Implementation of machine-learning classification in remote sensing: an applied review. *International Journal of Remote Sensing*, 39(9), 2784-2817. https://doi.org/10.1080/01431161.2018.1433343